# 步骤 1：微调模型（Fine-tuning）

## 练习目标

本笔记本是作者在 **Google Colab** 上完成的第 7 周工作的完整副本：用 **QLoRA** 对 `Llama-3.2-3B` 做商品价格预测的监督微调（SFT），再加载适配器做测试评估。

## 和本课第 7 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| QLoRA / 4-bit 量化 | `BitsAndBytesConfig` |
| PEFT LoRA | `LoraConfig` / `PeftModel` |
| SFT 训练 | `SFTTrainer` + `SFTConfig` |
| Hub 推送与评估 | `push_to_hub` + `evaluate`（来自 `util.py`） |

## 怎么跑

1. 在 Colab 里开启 GPU，并配置 Secrets：`HF_TOKEN`、`WANDB_API_KEY`
2. 从上到下运行；训练格会真正微调，耗时与算力相关
3. 后半段「测试」会按固定 `REVISION` 从 Hub 拉已训好的适配器


In [ ]:
# ========== 依赖安装：QLoRA / TRL 版本钉死，并拉取课程 util ==========
# 升级 bitsandbytes 与 trl 到与课程一致的版本（字符串/版本号必须保持原样）
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1
# 下载 week7/util.py（含 evaluate 等辅助函数），保存为本地 util.py
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py


In [ ]:
# ========== 导入：训练、登录、量化、可视化要用的库 ==========

# 标准库 os：写 WANDB_* 等环境变量
import os
# 标准库 re：正则（util / 后续可能用到）
import re
# 标准库 math：数值计算辅助
import math
# tqdm：进度条
from tqdm import tqdm
# Colab userdata：从 Secrets 读 HF_TOKEN / WANDB_API_KEY，避免写进代码
from google.colab import userdata
# Hugging Face Hub 登录
from huggingface_hub import login
# PyTorch：张量与 CUDA
import torch
# transformers 包本体（部分环境需要显式 import）
import transformers
# 因果 LM、分词器、训练参数类、设种子、BitsAndBytes 量化配置
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
# datasets：加载 Hub 上的 prompt 数据集；Dataset / DatasetDict 类型
from datasets import load_dataset, Dataset, DatasetDict
# Weights & Biases：实验追踪
import wandb
# PEFT：LoRA 配置
from peft import LoraConfig
# TRL：监督微调 Trainer 与配置
from trl import SFTTrainer, SFTConfig
# datetime：给这次 run 起带时间戳的名字
from datetime import datetime
# matplotlib：画图（评估阶段可能用到）
import matplotlib.pyplot as plt


True

In [ ]:
# ========== 常量与超参数：LITE_MODE 控制数据规模与 LoRA 强度 ==========

# 底座模型 id（必须保持英文原样，才能从 Hub 拉到正确权重）
BASE_MODEL = "meta-llama/Llama-3.2-3B"
# W&B / 项目名
PROJECT_NAME = "price"
# 你的 Hugging Face 用户名（推送模型用）；注释提醒改成自己的
HF_USER = "kicaronaldokello" # your HF name here!

# True=轻量试验；False=全量数据与更大 LoRA
LITE_MODE = True

# 数据集所有者
DATA_USER = "ed-donner"
# 按 LITE_MODE 在 lite / full 数据集名之间切换（字符串必须保持原样）
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

# 本次 run 名：时间戳；lite 时再加后缀
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
  RUN_NAME += "-lite"
# 本地输出目录 / 项目-run 组合名
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
# Hub 上的完整模型 id：用户名/项目-run
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# ----- 总体超参数 -----

# 训练轮数：lite 1，全量 3
EPOCHS = 1 if LITE_MODE else 3
# 每设备 batch：lite 较小以省显存
BATCH_SIZE = 32 if LITE_MODE else 256
# 单条序列最大 token 长度
MAX_SEQUENCE_LENGTH = 128
# 梯度累积步数（等效更大 batch）
GRADIENT_ACCUMULATION_STEPS = 1

# ----- QLoRA 超参数 -----

# True=4-bit；False 走 8-bit 分支
QUANT_4_BIT = True
# LoRA 秩 r
LORA_R = 32 if LITE_MODE else 256
# LoRA alpha：常取 2*r
LORA_ALPHA = LORA_R * 2
# 注意力投影层名（要挂 LoRA 的模块）
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
# MLP 层名；全量模式会一并加入 target_modules
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
# lite 只训注意力；全量=注意力+MLP
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
# LoRA dropout
LORA_DROPOUT = 0.1

# ----- 训练超参数 -----

# 学习率
LEARNING_RATE = 1e-4
# 预热比例
WARMUP_RATIO = 0.01
# 学习率调度类型（字符串必须保持原样）
LR_SCHEDULER_TYPE = 'cosine'
# 权重衰减
WEIGHT_DECAY = 0.001
# 优化器名（paged adamw，适配量化训练）
OPTIMIZER = "paged_adamw_32bit"

# 读 GPU 算力主版本号：>=8（Ampere+）才适合 bf16
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# ----- 日志 / 验证规模 -----

# 验证集截取条数
VAL_SIZE = 500 if LITE_MODE else 1000
# 打日志间隔（步）
LOG_STEPS = 5 if LITE_MODE else 10
# 存盘 / 评估间隔（步）
SAVE_STEPS = 100 if LITE_MODE else 200
# 是否上报 W&B
LOG_TO_WANDB = True


In [ ]:
# ========== 登录 Hugging Face：用 Colab Secret 取 token ==========

# 从 Colab Secrets 读取 HF_TOKEN（密钥名必须保持原样）
hf_token = userdata.get('HF_TOKEN')
# 登录 Hub；add_to_git_credential=True 便于后续 git/push
login(hf_token, add_to_git_credential=True)


In [ ]:
# ========== 登录并配置 Weights & Biases ==========

# 从 Secrets 读 W&B API Key
wandb_api_key = userdata.get('WANDB_API_KEY')
# 写入环境变量，供 wandb 客户端使用
os.environ["WANDB_API_KEY"] = wandb_api_key
# 执行登录
wandb.login()

# 把实验挂到 PROJECT_NAME；不自动上传整模；不 watch 梯度
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"


In [ ]:
# ========== 加载数据集：train / val / test ==========

# 按 DATASET_NAME 从 Hub 拉取
dataset = load_dataset(DATASET_NAME)
# 训练集
train = dataset['train']
# 验证集截到 VAL_SIZE 条，加快评估
val = dataset['val'].select(range(VAL_SIZE))
# 测试集（后半段评估用）
test = dataset['test']


In [ ]:
# ========== 初始化 W&B run（仅当开关打开） ==========

# LOG_TO_WANDB=True 时才 init，避免无密钥环境报错
if LOG_TO_WANDB:
  # project / name 与上面常量一致，便于在面板里找到本次 run
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)


In [71]:
# ========== 选择量化配置：4-bit NF4 或 8-bit ==========

# QUANT_4_BIT=True 走 QLoRA 常用 4-bit；否则 8-bit
if QUANT_4_BIT:
  # NF4 + double quant；计算用 bfloat16
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )
else:
  # 8-bit 备选路径
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )


In [ ]:
# ========== 加载分词器与 4/8-bit 底座模型 ==========

# 与 BASE_MODEL 匹配的 tokenizer；trust_remote_code 允许自定义代码
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# 没有 pad_token 时用 eos 顶上，否则 batch padding 会出问题
tokenizer.pad_token = tokenizer.eos_token
# 因果 LM 微调常见：右侧 padding
tokenizer.padding_side = "right"

# 按 quant_config 加载因果 LM；device_map=auto 自动分配设备
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
# 生成配置里的 pad_token_id 与 tokenizer 对齐
base_model.generation_config.pad_token_id = tokenizer.pad_token_id


In [66]:
# ========== LoRA 参数：只训练少量适配器权重 ==========

# LoraConfig：挂到 TARGET_MODULES 上的低秩适配器
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)


In [ ]:
# ========== SFTConfig：训练/日志/Hub/评估策略 ==========

# 监督微调超参；多数字段直接引用上面的常量
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    # 有 bf16 就开 bf16，否则用 fp16
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    # 开关打开才 report 到 wandb
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS
)


In [ ]:
# ========== 组装 SFTTrainer：模型 + 数据 + LoRA + 训练参数 ==========

# TRL 的监督微调 Trainer；peft_config 注入 LoRA
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters
)


In [ ]:
# ========== 真正开始微调，并把适配器推到 Hub ==========

# 启动训练循环（耗时取决于数据量与 GPU）
fine_tuning.train()

# 把微调后的（PEFT）模型推到 Hugging Face；private=True 私有仓库
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
# 打印 Hub 上的保存位置提示
print(f"Saved to the hub: {PROJECT_RUN_NAME}")


In [ ]:
# ========== 结束 W&B run ==========

# 若开了日志，收尾关闭 run，避免面板上一直显示 running
if LOG_TO_WANDB:
  wandb.finish()


# 步骤 2：测试已微调模型

下面重新设定常量（含固定 `RUN_NAME` / `REVISION`），从 Hub 加载适配器，用 `evaluate` 在测试集上算误差。


In [ ]:
# ========== 测试阶段常量：固定某次已训好的 run / revision ==========

# 底座模型（与训练时一致）
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
# Hub 上的用户名（适配器仓库归属）
HF_USER = "kicaronaldokello"

# 仍用 lite 数据做评估
LITE_MODE = True

# 注意：此处 DATA_USER 变量名保留，但 DATASET_NAME 字面量仍指向 ed-donner 数据
DATA_USER = "kicaronaldokello"
# lite / full 数据集名（字符串必须保持原样）
DATASET_NAME = f"ed-donner/items_prompts_lite" if LITE_MODE else f"ed-donner/items_prompts_full"

# 锁定某次训练产出的 run 名与 git revision，保证可复现加载
if LITE_MODE:
  RUN_NAME = "2026-03-07_16.13.22-lite"
  REVISION = "f352419fce90fde0d04b4534fb869d3ff1fd0e8c"
else:
  RUN_NAME = "2026-03-07_16.13.22"
  REVISION = "f352419fce90fde0d04b4534fb869d3ff1fd0e8c"

# 拼出项目-run 名与完整 Hub 模型 id
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# ----- QLoRA 相关：是否 4-bit，以及是否可用 bf16 -----

QUANT_4_BIT = True
# 再读一次 GPU capability，决定 bf16 / fp16
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8


In [ ]:
# ========== 只加载测试集 ==========

# 按 DATASET_NAME 拉取；评估只用 test split
dataset = load_dataset(DATASET_NAME)
test = dataset['test']


In [ ]:
# ========== 评估用量化配置：按 GPU 在 bf16 / fp16 间切换 ==========

# 与训练类似，但 compute dtype 会随 use_bf16 变化
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    # Ampere+ 用 bfloat16，否则 float16
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )


In [ ]:
# ========== 加载底座 + 从 Hub 挂上 PEFT 适配器 ==========

# 分词器与训练时同一 BASE_MODEL
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 量化加载底座
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# 有 REVISION 就按 commit 拉取，保证与记录指标一致；否则用默认分支
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)


# 打印显存占用（MB），确认量化生效
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")


In [ ]:
# ========== 在笔记本里展示模型对象（确认加载成功） ==========

# 表达式语句：Notebook 会显示 PeftModel 摘要
fine_tuned_model


In [ ]:
# ========== 推理函数：对单条 item 生成续写并 decode ==========

def model_predict(item):
    # 把 prompt 编成张量并放到 CUDA
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    # 推理不需要梯度，省显存
    with torch.no_grad():
        # 最多新生成 8 个 token（价格很短）
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    # 只取「新增」部分，去掉 prompt 对应的 token
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    # 解码成字符串，交给 evaluate 去抽价格
    return tokenizer.decode(generated_ids)


In [ ]:
# ========== 固定种子后跑课程 evaluate ==========

# 固定随机性，便于对比
set_seed(42)
# evaluate 来自 util.py：对 test 集调用 model_predict 并汇总误差
evaluate(model_predict, test)


## 模型预测结果（作者记录）

在作者那次 run 上观测到的指标（供对照，不是你本机必须复现的数字）：

- **Error（平均绝对误差 MAE）：** $65.12
- **MSE：** 13,813
- **r²：** 37.2%
